# Simplified self attention

### Simple self attention without trainable weight

The goal of self attention is to compute ta context vector for each input element that combines information from all other input elements

- A context vector can be represented as an enriched embedding vector
- Context vector's role in self attention is crucial - to create n enriched represnetations of each element in the input sequence by incorporating information from all other elements in the sequence

First step in implementing self attention is comupting attention scores

In [4]:
import torch

inputs = torch.tensor(
    [[0.43, 0.15, 0.89], # Your (x^1)
     [0.55, 0.87, 0.66], # Journey (x^2)
     [0.57, 0.85, 0.64], # Starts (x^3)
     [0.22, 0.58, 0.33], # With (x^4)
     [0.77, 0.25, 0.10], # One (x^5)
     [0.05, 0.80, 0.55]] # Step (x^6)
)

In [7]:
attention_scores_2 = torch.empty(inputs.shape[0])
attention_scores_2

tensor([0., 0., 0., 0., 0., 0.])

In [ ]:
query =inputs[2] # (x^2)
query 

tensor([0.5700, 0.8500, 0.6400])

In [10]:
for i, x in enumerate(inputs):
    attention_scores_2[i] = torch.dot(x, query)
attention_scores_2


tensor([0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605])

- The dot product is a measure of similarity becuase it quantifies how closely two vectors are aligned, a higher dot product indicates a greater degree of alignment or similarity between vectors
- in self attention  dot product determines the extent to which each element in a sequence focuses on or attends to any other element
- higher dot product, higher similarity and attention score between two elements

In [11]:
attention_weights_2_temp = attention_scores_2 / attention_scores_2.sum()
print("Attention scores: ", attention_scores_2)
print("Attention weights: ", attention_weights_2_temp)
print("Sum of attention weights: ", attention_weights_2_temp.sum())


Attention scores:  tensor([0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605])
Attention weights:  tensor([0.1454, 0.2277, 0.2248, 0.1280, 0.1104, 0.1637])
Sum of attention weights:  tensor(1.)


In practice it is much advisable to use softmax for normaliztion

Normalization provides a better way of managing exteme values offer more favorable gradient properties during training

In [12]:
def softmax(x):
    exp_x = torch.exp(x)
    sum_exp_x = torch.sum(exp_x, dim=0)
    return exp_x / sum_exp_x

attention_weights_2_naive = softmax(attention_scores_2)
print("Attention weights: ", attention_weights_2_naive)
print("Sum of attention weights: ", attention_weights_2_naive.sum())


Attention weights:  tensor([0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565])
Sum of attention weights:  tensor(1.)


Softmax function also ensures that attention weigths come out positieve

This ensure better interpretation of attention weights as probabilities or rrelative improtance where higher weights indicate greater importance

In [13]:
# using pytorch softmax to avoid instability problems

attention_weights_2 = torch.softmax(attention_scores_2, dim=0)
print("Attention weights: ", attention_weights_2)
print("Sum of attention weights: ", attention_weights_2.sum())


Attention weights:  tensor([0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565])
Sum of attention weights:  tensor(1.0000)


Context vector is calculated by multiplying the embedded input tokens with corresponding attention weights and then summing the resulting vectors

Basically a weighted sum of all input vectors, attention weights lenght here obviously will be equal to the lenght of the inputs

In [27]:
# compute context vector

query= inputs[1] # (x^2)
context_vector_2  = torch.zeros(query.shape)
for i, x in enumerate(inputs):
    print("Context vector before addition: ", context_vector_2)
    print("Attention weight of the input: ", attention_weights_2[i])
    print("Input vector: ", x)
    print("Weighted input vector: ", attention_weights_2[i] * x)
    context_vector_2  =  context_vector_2 + attention_weights_2[i] * x
    print("context_vector_2: ", context_vector_2)
    print("--------------------------------")

print("Context vector: ", context_vector_2)

Context vector before addition:  tensor([0., 0., 0.])
Attention weight of the input:  tensor(0.1390)
Input vector:  tensor([0.4300, 0.1500, 0.8900])
Weighted input vector:  tensor([0.0598, 0.0209, 0.1237])
context_vector_2:  tensor([0.0598, 0.0209, 0.1237])
--------------------------------
Context vector before addition:  tensor([0.0598, 0.0209, 0.1237])
Attention weight of the input:  tensor(0.2369)
Input vector:  tensor([0.5500, 0.8700, 0.6600])
Weighted input vector:  tensor([0.1303, 0.2061, 0.1564])
context_vector_2:  tensor([0.1901, 0.2270, 0.2801])
--------------------------------
Context vector before addition:  tensor([0.1901, 0.2270, 0.2801])
Attention weight of the input:  tensor(0.2326)
Input vector:  tensor([0.5700, 0.8500, 0.6400])
Weighted input vector:  tensor([0.1326, 0.1977, 0.1489])
context_vector_2:  tensor([0.3227, 0.4247, 0.4290])
--------------------------------
Context vector before addition:  tensor([0.3227, 0.4247, 0.4290])
Attention weight of the input:  tenso

### Computing self attention for all input tokens

In [28]:
import torch

inputs = torch.tensor(
    [[0.43, 0.15, 0.89], # Your (x^1)
     [0.55, 0.87, 0.66], # Journey (x^2)
     [0.57, 0.85, 0.64], # Starts (x^3)
     [0.22, 0.58, 0.33], # With (x^4)
     [0.77, 0.25, 0.10], # One (x^5)
     [0.05, 0.80, 0.55]] # Step (x^6)
)

Need to compute attention scores for all input tokens so the attnetions cores matrix need to be able to store attentions cores of each inptu query with respect to the other inputs

In [31]:
attention_scores = torch.empty(inputs.shape[0], inputs.shape[0]) 
attention_scores.shape


torch.Size([6, 6])

In [32]:
for i, x_i in enumerate(inputs):
    for j, x_j in enumerate(inputs):
        attention_scores[i, j] = torch.dot(x_i, x_j)
attention_scores

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])

can also use matrix multiplication

In [37]:
inputs

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])

In [38]:
inputs.T

tensor([[0.4300, 0.5500, 0.5700, 0.2200, 0.7700, 0.0500],
        [0.1500, 0.8700, 0.8500, 0.5800, 0.2500, 0.8000],
        [0.8900, 0.6600, 0.6400, 0.3300, 0.1000, 0.5500]])

In [40]:
attention_scores = inputs @ inputs.T
attention_scores

tensor([[0.9995, 0.9544, 0.9422, 0.4753, 0.4576, 0.6310],
        [0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865],
        [0.9422, 1.4754, 1.4570, 0.8296, 0.7154, 1.0605],
        [0.4753, 0.8434, 0.8296, 0.4937, 0.3474, 0.6565],
        [0.4576, 0.7070, 0.7154, 0.3474, 0.6654, 0.2935],
        [0.6310, 1.0865, 1.0605, 0.6565, 0.2935, 0.9450]])

In [33]:
# compute attention weights

attention_weights = torch.softmax(attention_scores, dim=1)
attention_weights

tensor([[0.2098, 0.2006, 0.1981, 0.1242, 0.1220, 0.1452],
        [0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581],
        [0.1390, 0.2369, 0.2326, 0.1242, 0.1108, 0.1565],
        [0.1435, 0.2074, 0.2046, 0.1462, 0.1263, 0.1720],
        [0.1526, 0.1958, 0.1975, 0.1367, 0.1879, 0.1295],
        [0.1385, 0.2184, 0.2128, 0.1420, 0.0988, 0.1896]])

In [36]:
# verify that the sum of attention weights for each input is 1
attention_weights.sum(dim=1)

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000])

In [41]:
# compute context scores

context_vectors = attention_weights @ inputs
context_vectors

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

# Implementing self attention with trainable weights

Almost similar to the previous implementation with slight differences with most notable being intorduction to weight matrices that are updated during model training

Introducing Wq, Wk, Wv matrices 3 trainable weigth matrices used to project embedded input tokens to query, key and value vectors

In [55]:
x_2 = inputs[1]
d_in = inputs.shape[1]
d_out = 2

In [56]:
torch.manual_seed(123)
Wq = torch.nn.Parameter(torch.randn(d_in, d_out, requires_grad=True))
Wk = torch.nn.Parameter(torch.randn(d_in, d_out, requires_grad=True))
Wv = torch.nn.Parameter(torch.randn(d_in, d_out, requires_grad=True))

In [57]:
print("Wq: ", Wq)
print("x_2: ", x_2)
query_2 =  x_2 @ Wq
print("query_2: ", query_2)

Wq:  Parameter containing:
tensor([[-0.1115,  0.1204],
        [-0.3696, -0.2404],
        [-1.1969,  0.2093]], requires_grad=True)
x_2:  tensor([0.5500, 0.8700, 0.6600])
query_2:  tensor([-1.1729, -0.0048], grad_fn=<SqueezeBackward4>)


In [58]:
key_2 = x_2 @ Wk
print("key_2: ", key_2)
value_2 = x_2 @ Wv
print("value_2: ", value_2)

key_2:  tensor([-0.1142, -0.7676], grad_fn=<SqueezeBackward4>)
value_2:  tensor([0.4107, 0.6274], grad_fn=<SqueezeBackward4>)


### Weight parameters vs Attention weights

- Weigth matrieces W is short for "weigth parameters" -  the values neural netwroks optimized during training
- Attention weights is determine the extent to which context vector depends on the different parts of the input ( to what extent netwrok depends on parts of the input)

- Weight Parameters are fundamental learned coefficients that define the netwroks connections
- Attention weights are dynamic context specific values

Even tough we want to compute context vector for only the second input (z2/context_vector_2)  we will still nead key and value of all inputs to compute the attention weights

In [61]:
torch.manual_seed(123)
keys = inputs @ Wk
values = inputs @ Wv
print("keys: ", keys)
print("values: ", values)

keys:  tensor([[-0.1823, -0.6888],
        [-0.1142, -0.7676],
        [-0.1443, -0.7728],
        [ 0.0434, -0.3580],
        [-0.6467, -0.6476],
        [ 0.3262, -0.3395]], grad_fn=<MmBackward0>)
values:  tensor([[ 0.1196, -0.3566],
        [ 0.4107,  0.6274],
        [ 0.4091,  0.6390],
        [ 0.2436,  0.4182],
        [ 0.2653,  0.6668],
        [ 0.2728,  0.3242]], grad_fn=<MmBackward0>)


In [64]:
print("keys.shape: ", keys.shape)
print("values.shape: ", values.shape)
print("query_2.shape: ", query_2.shape)

keys.shape:  torch.Size([6, 2])
values.shape:  torch.Size([6, 2])
query_2.shape:  torch.Size([2])


In [67]:
keys_2 = keys[1]
keys_2

tensor([-0.1142, -0.7676], grad_fn=<SelectBackward0>)

In [68]:
attention_score_22 = query_2.dot(keys_2)
print("attention_score_22: ", attention_score_22)

attention_score_22:  tensor(0.1376, grad_fn=<DotBackward0>)


In [65]:
attention_scores_2 = query_2 @ keys.T
print("attention_scores_2: ", attention_scores_2)

attention_scores_2:  tensor([ 0.2172,  0.1376,  0.1730, -0.0491,  0.7616, -0.3809],
       grad_fn=<SqueezeBackward4>)


To compute attention weights, we apply softmax to the attention scores after scaling them by the square root of the key's embedding dimension. 
This scaling is important because it helps prevent the softmax from having extremely small gradients when the dimensionality is large, 
making optimization more stable.

Example: If the attention scores for a query are [2.0, 1.0, 0.1] and the embedding dimension (d_k) is 4, 
then we divide each score by sqrt(4) = 2 and apply softmax to [1.0, 0.5, 0.05].

In [73]:
keys

tensor([[-0.1823, -0.6888],
        [-0.1142, -0.7676],
        [-0.1443, -0.7728],
        [ 0.0434, -0.3580],
        [-0.6467, -0.6476],
        [ 0.3262, -0.3395]], grad_fn=<MmBackward0>)

In [74]:
keys.shape

torch.Size([6, 2])

In [75]:
d_k = keys.shape[1]

In [76]:
attention_weights_2  = torch.softmax(attention_scores_2/ d_k**0.5, dim=-1)
attention_weights_2

tensor([0.1704, 0.1611, 0.1652, 0.1412, 0.2505, 0.1117],
       grad_fn=<SoftmaxBackward0>)

In [77]:
# verify that the sum of attention weights for each input is 1
attention_weights_2.sum(dim=-1)

tensor(1.0000, grad_fn=<SumBackward1>)

Compute context vector by multiplying value vectors with corresponding attention weight

In [78]:
context_vector_2  =  attention_weights_2 @ values
context_vector_2

tensor([0.2854, 0.4081], grad_fn=<SqueezeBackward4>)

### Implementing a compact self attention python class

In [104]:
import torch.nn as nn

class SelfAttention_V1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.d_in = d_in
        self.d_out = d_out
        self.Wq = nn.Parameter(torch.randn(d_in, d_out, requires_grad=True))
        self.Wk = nn.Parameter(torch.randn(d_in, d_out, requires_grad=True))
        self.Wv = nn.Parameter(torch.randn(d_in, d_out, requires_grad=True))

    def forward(self, x):
        keys = x @ self.Wk
        print("keys: ", keys.shape)
        values = x @ self.Wv
        query = x @ self.Wq
        attention_scores = query @ keys.T
        d_k = keys.shape[-1]
        attention_weights = torch.softmax(attention_scores/ d_k**0.5, dim=-1)
        context_vector = attention_weights @ values
        return context_vector

In [105]:
import torch

inputs = torch.tensor(
    [[0.43, 0.15, 0.89], # Your (x^1)
     [0.55, 0.87, 0.66], # Journey (x^2)
     [0.57, 0.85, 0.64], # Starts (x^3)
     [0.22, 0.58, 0.33], # With (x^4)
     [0.77, 0.25, 0.10], # One (x^5)
     [0.05, 0.80, 0.55]] # Step (x^6)
)

In [106]:
d_in = inputs.shape[1]
d_out = 2
torch.manual_seed(123)
self_attention_v1 = SelfAttention_V1(d_in, d_out)
context_vectors = self_attention_v1(inputs)
context_vectors


keys:  torch.Size([6, 2])


tensor([[0.2845, 0.4071],
        [0.2854, 0.4081],
        [0.2854, 0.4075],
        [0.2864, 0.3974],
        [0.2863, 0.3910],
        [0.2860, 0.4039]], grad_fn=<MmBackward0>)

Improvement: Use nn.Linear instead of nn.Parameter for Wq, Wk, Wv.
 
Reason: nn.Linear implements a learnable linear transformation (with bias by default),
taking care of weight initialization, bias addition, and providing more modularity.
It is easier to use, especially if you want to change input/output sizes,
and integrates seamlessly with other nn.Module features like .parameters() and device management.
 Using nn.Linear also makes code more readable and concise.

 nn.Linear uses a more sophisticated weight initilaization scheme and will yeild different outputs

In [ ]:
import torch.nn as nn

class SelfAttention_V2(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.d_in = d_in
        self.d_out = d_out
        self.Wq = nn.Linear(d_in, d_out, bias=False)
        self.Wk = nn.Linear(d_in, d_out, bias=False)
        self.Wv = nn.Linear(d_in, d_out, bias=False)

    def forward(self, x):
        keys = self.Wk(x) # performs matrix multiplication when bias is False
        print("keys: ", keys.shape)
        values = self.Wv(x)
        query = self.Wq(x)
        attention_scores = query @ keys.T
        d_k = keys.shape[-1]
        attention_weights = torch.softmax(attention_scores/ d_k**0.5, dim=-1)
        context_vector = attention_weights @ values
        return context_vector

In [108]:
inputs

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])

In [109]:
d_in = inputs.shape[1]
d_out = 2
torch.manual_seed(789)
self_attention_v2 = SelfAttention_V2(d_in, d_out)
context_vectors = self_attention_v2(inputs)
context_vectors


keys:  torch.Size([6, 2])


tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)

In [113]:
Weight_q_v2 = self_attention_v2.Wq.weight.T
Weight_k_v2 = self_attention_v2.Wk.weight.T
Weight_v_v2 = self_attention_v2.Wv.weight.T
print("Weight_q_v2: ", Weight_q_v2)
print("Weight_k_v2: ", Weight_k_v2)
print("Weight_v_v2: ", Weight_v_v2)


Weight_q_v2:  tensor([[ 0.3161, -0.1683],
        [ 0.4568, -0.3379],
        [ 0.5118, -0.0918]], grad_fn=<PermuteBackward0>)
Weight_k_v2:  tensor([[ 0.4058,  0.2134],
        [-0.4704, -0.2601],
        [ 0.2368, -0.5105]], grad_fn=<PermuteBackward0>)
Weight_v_v2:  tensor([[ 0.2526,  0.5191],
        [-0.1415, -0.0852],
        [-0.1962, -0.2043]], grad_fn=<PermuteBackward0>)


In [115]:
import torch.nn as nn

class SelfAttention_V1_fused_weights_V2(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.d_in = d_in
        self.d_out = d_out
        self.Wq = nn.Parameter(torch.randn(d_in, d_out, requires_grad=True))
        self.Wk = nn.Parameter(torch.randn(d_in, d_out, requires_grad=True))
        self.Wv = nn.Parameter(torch.randn(d_in, d_out, requires_grad=True))

    def forward(self, x, wq, wk, wv):
        keys = x @ wk
        print("keys: ", keys.shape)
        values = x @ wv
        query = x @ wq
        attention_scores = query @ keys.T
        d_k = keys.shape[-1]
        attention_weights = torch.softmax(attention_scores/ d_k**0.5, dim=-1)
        context_vector = attention_weights @ values
        return context_vector

In [116]:
self_attention_v1_fused_weights_v2 = SelfAttention_V1_fused_weights_V2(d_in, d_out)
context_vectors = self_attention_v1_fused_weights_v2(inputs, Weight_q_v2, Weight_k_v2, Weight_v_v2)
context_vectors

keys:  torch.Size([6, 2])


tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)

# Casual Attention

or masked attention is a form of self attention in which we only consider the previous and current inputs in sequence when processing an give token  when computing attention scores

we mask out the attention weights above the diagonal and we normalize the non masked attention weights such that attention weigths sum to 1 in each row

In [117]:
inputs

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])

In [119]:
queries = self_attention_v2.Wq(inputs)
queries

tensor([[ 0.6600, -0.2047],
        [ 0.9091, -0.4471],
        [ 0.8960, -0.4419],
        [ 0.5034, -0.2633],
        [ 0.4088, -0.2232],
        [ 0.6628, -0.3292]], grad_fn=<MmBackward0>)

In [121]:
keys = self_attention_v2.Wk(inputs)
keys

tensor([[ 0.3147, -0.4016],
        [-0.0298, -0.4459],
        [-0.0170, -0.4262],
        [-0.1054, -0.2724],
        [ 0.2185,  0.0482],
        [-0.2258, -0.4782]], grad_fn=<MmBackward0>)

In [122]:
attention_scores = queries @ keys.T
attention_scores

tensor([[ 0.2899,  0.0716,  0.0760, -0.0138,  0.1344, -0.0511],
        [ 0.4656,  0.1723,  0.1751,  0.0259,  0.1771,  0.0085],
        [ 0.4594,  0.1703,  0.1731,  0.0259,  0.1745,  0.0090],
        [ 0.2642,  0.1024,  0.1036,  0.0186,  0.0973,  0.0122],
        [ 0.2183,  0.0874,  0.0882,  0.0177,  0.0786,  0.0144],
        [ 0.3408,  0.1270,  0.1290,  0.0198,  0.1290,  0.0078]],
       grad_fn=<MmBackward0>)

In [124]:
attention_weights = torch.softmax(attention_scores/ d_k**0.5, dim=-1)
attention_weights

tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)

In [127]:
context_length = inputs.shape[0]
mask = torch.tril(torch.ones(context_length, context_length))
print(mask)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [138]:
masked_attention_weights =  attention_weights * mask
masked_attention_weights

tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)

In [140]:
normalized_masked_attention_weights = masked_attention_weights / masked_attention_weights.sum(dim=-1, keepdim=True)
normalized_masked_attention_weights

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<DivBackward0>)

In [141]:
#verify that the sum of attention weights for each input is 1
normalized_masked_attention_weights.sum(dim=-1)

tensor([1.0000, 1.0000, 1.0000, 1.0000, 1.0000, 1.0000],
       grad_fn=<SumBackward1>)

it might appear the using softmax on masked attention weight might lead to information leakage since the future toekns are present in the denominator in the softmax function but after calculating the attention weight when we mask and re nromalized we are essntionlly nullifying the effect of masked tokens

A more efficient way is to obtain the masked attention weigth matrix in casual attention is to mask the attention scores with negative infinity values before applying the softmax function

Since softmax converts the inputs to probabilities. -inf becomes 0 (e^-inf =0) 

In [ ]:
context_length = inputs.shape[0]
torch.ones(context_length, context_length)

tensor([[1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.],
        [1., 1., 1., 1., 1., 1.]])

In [146]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
mask

tensor([[0., 1., 1., 1., 1., 1.],
        [0., 0., 1., 1., 1., 1.],
        [0., 0., 0., 1., 1., 1.],
        [0., 0., 0., 0., 1., 1.],
        [0., 0., 0., 0., 0., 1.],
        [0., 0., 0., 0., 0., 0.]])

In [147]:
masked_attention_scores  = attention_scores.masked_fill(mask.bool(), float('-inf'))
masked_attention_scores

tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)

In [148]:
masked_attention_weights = torch.softmax(masked_attention_scores/ d_k**0.5, dim=-1)
masked_attention_weights


tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)

### Dropout

- Dropout is a technique where randomly selected hidden layers are ignored during training, hence "dropping" them out
- Helpo prevent overfitting by ensuringmodel is not overly reliant on a specific set of hidden layers

- In Attention mechanism typically applie at two specific times:
    - after calculating attention weights
    - or after applying attention weights to value vectors

- dropout rate of 50% means masking out half of the attention weights
- to compensate for the reductionin in active elements, the values of the remaining elements in the matrix are scaled up by a factor of 1/0.5

In [149]:
torch.manual_seed(123)
dropout = torch.nn.Dropout(p=0.5)
example = torch.ones(4,4)
print("Example: ", example)
example_dropped = dropout(example)
print("Example dropped: ", example_dropped)


Example:  tensor([[1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.],
        [1., 1., 1., 1.]])
Example dropped:  tensor([[2., 2., 0., 2.],
        [2., 0., 0., 0.],
        [0., 2., 0., 2.],
        [2., 2., 2., 2.]])


applying dropout to the attention weigth matrix

In [150]:
torch.manual_seed(123)
print( dropout(masked_attention_weights))


tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.7599, 0.6194, 0.6206, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4921, 0.4925, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3966, 0.0000, 0.3775, 0.0000, 0.0000],
        [0.0000, 0.3327, 0.3331, 0.3084, 0.3331, 0.0000]],
       grad_fn=<MulBackward0>)


### Implementing a compact class for causal attention

In [204]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length):
        super().__init__()
        self.d_in = d_in
        self.d_out = d_out
        self.Wq = nn.Linear(d_in, d_out, bias=False)
        self.Wk = nn.Linear(d_in, d_out, bias=False)
        self.Wv = nn.Linear(d_in, d_out, bias=False)
        self.dropout = nn.Dropout(p=0.5)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length)))
    
    def forward(self, x):
        keys = self.Wk(x)
        values = self.Wv(x)
        query = self.Wq(x)
        b, t, c = x.shape
        attention_scores = query @ keys.transpose(-2, -1) # (b, t, c) @ (b, c, t) -> (b, t, t) transpose only dimension 1 and 2 keeping batch dimension same
        attention_scores.masked_fill_(mask.bool()[:t, : t], float('-inf'))
        masked_attention_weights = torch.softmax(attention_scores/ keys.shape[-1]**0.5, dim=-1)
        masked_attention_weights = self.dropout(masked_attention_weights)
        context_vector = masked_attention_weights @ values
        return context_vector

In [205]:
inputs

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])

In [206]:
batch = torch.stack([inputs, inputs])
batch

tensor([[[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]],

        [[0.4300, 0.1500, 0.8900],
         [0.5500, 0.8700, 0.6600],
         [0.5700, 0.8500, 0.6400],
         [0.2200, 0.5800, 0.3300],
         [0.7700, 0.2500, 0.1000],
         [0.0500, 0.8000, 0.5500]]])

In [207]:
torch.manual_seed(123)
causal_attention = CausalAttention(d_in=3, d_out=2, context_length=batch.shape[1])
context_vectors = causal_attention(batch)
context_vectors

tensor([[[-0.9038,  0.4432],
         [-0.4368,  0.2142],
         [-0.4849, -0.1341],
         [-0.5834,  0.0081],
         [-0.6219, -0.0526],
         [-0.1417, -0.0505]],

        [[ 0.0000,  0.0000],
         [-1.1749,  0.0116],
         [-0.7733,  0.0073],
         [-0.9140, -0.2769],
         [-0.7679, -0.0735],
         [-0.6749, -0.0984]]], grad_fn=<UnsafeViewBackward0>)

# Multi Head Attention

Implementing mul.ti head attenntions involves creating multiple instances of causal attention mechanism , each with its own weights and then combining their outputs

The main idea behind the multi head attention is to run the attention mechanism multiple times (in parallel) with different learned linear projectsions -  the results of multiplying the input data (like query, key and value vectors) by a wieght matrix

In [210]:
# naive implementation by running causal attention multiple times
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, num_heads, context_length, qkv_bias=False, dropout=0.5):
        super().__init__()
        self.heads = nn.ModuleList([CausalAttention(d_in, d_out, context_length) for _ in range(num_heads)])

    def forward(self, x):
        out = torch.cat([head(x) for head in self.heads], dim=-1)
        return out
        

In [211]:
torch.manual_seed(123)
multi_head_attention = MultiHeadAttentionWrapper(d_in=3, d_out=2, num_heads=2, context_length=batch.shape[1])
multi_head_attention(batch)

tensor([[[ 0.0000,  0.0000,  0.0000,  0.0000],
         [-0.7381, -0.2026,  0.0000,  0.0000],
         [-0.9717, -0.2678,  0.9703,  0.7117],
         [-0.2210,  0.1084,  0.3492,  0.2569],
         [-0.4833, -0.1435,  0.4143,  0.3235],
         [-0.9130, -0.2881,  0.8947,  0.5938]],

        [[ 0.0000,  0.0000,  0.9544,  0.2127],
         [ 0.0000,  0.0000,  1.1781,  0.6513],
         [-0.7751,  0.0077,  0.4856,  0.3552],
         [-0.5504, -0.1770,  0.8795,  0.6697],
         [-0.8117, -0.1150,  0.4682,  0.3227],
         [-0.4257, -0.1539,  0.6311,  0.4446]]], grad_fn=<CatBackward0>)

# Implementing a effcient Multi Head Attention class

head_dim is the out dimension width of one head

`.view` splits the last axis, `d_out`, into `num_heads` by `head_dim`. The number of values stays the same.

```python
# [2, 6, 2] -> [2, 6, 2, 1]
queries.view(batch, tokens, num_heads, head_dim)
```

With `d_out = 2` and 2 heads, `head_dim = 2 // 2 = 1`. The 2 numbers for one token become 1 number for each head. Joining the heads again gives width `d_out`.

In [228]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, num_heads, context_length, qkv_bias=False, dropout=0.5):
        super().__init__()

        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        self.d_in = d_in
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.Wq = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.Wk = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.Wv = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(p=dropout)
        self.out_proj = nn.Linear(d_out, d_out)
        self.register_buffer('mask', torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        batch_size, number_of_tokens, d_in = x.shape # (b, t, c)

        queries = self.Wq(x)
        keys = self.Wk(x)
        values = self.Wv(x)

        queries = queries.view(batch_size, number_of_tokens, self.num_heads, self.head_dim) # [b,t,c] -> [b,t,nh,hd]
        keys = keys.view(batch_size, number_of_tokens, self.num_heads, self.head_dim) # [b,t,c] -> [b,t,nh,hd]
        values = values.view(batch_size, number_of_tokens, self.num_heads, self.head_dim) # [b,t,c] -> [b,t,nh,hd]

        queries = queries.transpose(1, 2) # [b,t,nh,hd] -> [b,nh,t,hd]
        keys = keys.transpose(1, 2) # [b,t,nh,hd] -> [b,nh,t,hd]
        values = values.transpose(1, 2) # [b,t,nh,hd] -> [b,nh,t,hd]

        attention_scores = queries @ keys.transpose(-2, -1) # [b,nh,t,hd] @ [b,nh,hd,t] -> [b,nh,t,t]

        attention_scores.masked_fill_(self.mask.bool()[:number_of_tokens, :number_of_tokens], float('-inf'))

        attention_weights = torch.softmax(attention_scores/ keys.shape[-1]**0.5, dim=-1)
        attention_weights = self.dropout(attention_weights)

        context_vector = attention_weights @ values # [b,nh,t,t] @ [b,nh,t,hd] -> [b,nh,t,hd]
        context_vector = context_vector.transpose(1, 2) # [b,nh,t,hd] -> [b,t,nh,hd]
        context_vector = context_vector.contiguous().view(batch_size, number_of_tokens, self.d_out) # [b,t,nh,hd] -> [b,t,d_out]
        context_vector = self.out_proj(context_vector)
        return context_vector
        

In [229]:
inputs

tensor([[0.4300, 0.1500, 0.8900],
        [0.5500, 0.8700, 0.6600],
        [0.5700, 0.8500, 0.6400],
        [0.2200, 0.5800, 0.3300],
        [0.7700, 0.2500, 0.1000],
        [0.0500, 0.8000, 0.5500]])

In [230]:
batch = torch.stack([inputs, inputs])
batch.shape

torch.Size([2, 6, 3])

In [232]:
torch.manual_seed(123)
batch_size, number_of_tokens, d_in = batch.shape
d_out = 2
num_heads = 2
multi_head_attention = MultiHeadAttention(d_in, d_out, num_heads, context_length=number_of_tokens)
context_vectors = multi_head_attention(batch)
context_vectors

tensor([[[0.3441, 0.2306],
         [0.3178, 0.3094],
         [0.2752, 0.4373],
         [0.3305, 0.2162],
         [0.1766, 0.5667],
         [0.2615, 0.3981]],

        [[0.2940, 0.7409],
         [0.3680, 0.3385],
         [0.2487, 0.4232],
         [0.2927, 0.3046],
         [0.2103, 0.4023],
         [0.2399, 0.4054]]], grad_fn=<ViewBackward0>)

In [233]:
context_vectors.shape

torch.Size([2, 6, 2])

In [238]:
# gpt-2 style multi head attention

num_heads = 12
d_in = 768
d_out = 768
context_length = 1024
batch_size = 20
number_of_tokens = 1024

inputs = torch.randn(batch_size, number_of_tokens, d_in)

In [239]:
inputs.shape

torch.Size([20, 1024, 768])

In [240]:
torch.manual_seed(123)
multi_head_attention = MultiHeadAttention(d_in, d_out, num_heads, context_length=number_of_tokens)
context_vectors = multi_head_attention(inputs)
context_vectors.shape

torch.Size([20, 1024, 768])